# Contract

A contract is the base concept from which you can calculate energy costs.
It consists of 4 components:
- a `provider` tariff
- a `distributor` tariff
- `fees` — which are essentially government tariffs — multiple fee tariffs can be provided as a list and will be merged automatically
- a `tax_rate` (a percentage applied to all tariff costs)

Given a contract and consumption data, you can calculate the cost of that consumption under the terms of the contract.

The output DataFrame uses a structured MultiIndex with fixed, translatable labels based on `TariffCategory` and `CostGroup` enums, making it predictable and easy to use for i18n.

In [1]:
import pandas as pd

from energy_cost import Contract, Meter, Tariff
from energy_cost.data.be import distributors, fees, tax_rate

contract = Contract(
    provider=Tariff.from_yaml("../examples/tariffs/fixed.yml"),
    distributor=distributors["fluvius_imewo"],
    fees=[fees["flanders_residential"], fees["be_residential"]],
    tax_rate=tax_rate,
)

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption])

timestamp    provider               distributor             \
                            consumption         total consumption              
                                 energy  total  total      energy      total   
0 2024-01-01 00:00:00+01:00       59.52  59.52  59.52   23.676520  23.676520   
1 2024-02-01 00:00:00+01:00       55.68  55.68  55.68   22.149003  22.149003   
2 2024-03-01 00:00:00+01:00        0.02   0.02   0.02    0.007956   0.007956   

                                                         fees  \
   capacity            fixed                total consumption   
      total data_manangement     total      total      excise   
0  8.209764         1.209508  1.209508  33.095793   28.260096   
1  8.209764         1.131475  1.131475  31.490243   26.436864   
2  8.209764         1.209508  1.209508   9.427228    0.009496   

                                                taxes       total  
                                      total     total       total  
  energy_contribution      total      total     total       total  
0            1.146415  29.406511  29.406511  7.886495  139.328071  
1            1.072452  27.509316  27.509316  7.441248  131.462047  
2            0.000385   0.009881   0.009881  1.132583   20.008965

> Note: most tariffs are based on indexes, make sure to define all required indexes before calculating costs. See the [index documentation](./index.ipynb) for more details. If you are definingn your own tariffs, also have a look at the [tariff documentation](./tariff.ipynb) for details on how to implement tariffs.

### Resolution

By default the cost is calculated for each month, but you can specify a different resolution if you want to calculate costs for different time periods eg yearly.

In [2]:
import isodate

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption], resolution=isodate.Duration(years=1))

timestamp    provider                                   \
                            consumption             fixed          total   
                                 energy   total fixed_fee  total   total   
0 2024-01-01 00:00:00+01:00      702.72  702.72       0.0    0.0  702.72   
1 2025-01-01 00:00:00+01:00      630.72  630.72     120.0  120.0  750.72   
2 2026-01-01 00:00:00+01:00      135.96  135.96     180.0  180.0  315.96   

  distributor                                                              \
  consumption                capacity            fixed              total   
       energy       total       total data_manangement  total       total   
0  279.535692  279.535692   98.517173            14.28  14.28  392.332864   
1  412.792925  412.792925  133.105596            17.51  17.51  563.408521   
2   59.240491   59.240491   33.875614            17.85  17.85  110.966105   

         fees                                                   taxes  \
  consumption                                       total       total   
       excise energy_contribution       total       total       total   
0  333.651456           13.535090  347.186546  347.186546   93.302195   
1  332.739840           13.498109  346.237949  346.237949  115.858924   
2   53.794840            2.182271   55.977111   55.977111   42.877730   

         total  
         total  
         total  
0  1648.338778  
1  2046.840990  
2   757.506559

### Time range

By default start and end are set to the first and last timestamp in the data, but you can specify a different start and end if you want to calculate costs for a different time period than the one covered by the data.

This is useful for the Flemish capacity tariff that uses a rolling average of the consumption of the last 12 months to determine the cost for the next month. In this case you would need to provide data for at least 12 months before the start of the period you want to calculate costs for.

In [3]:
import datetime as dt
from zoneinfo import ZoneInfo

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": [0.002] * 4 * 24 * 365 + [0.003] * 4 * 24 * 365 + [0.003] * 4 * 24 * 60 + [0.003],
        }
    )
)

contract.calculate_cost(
    [consumption],
    start=dt.datetime(2025, 1, 1, tzinfo=ZoneInfo("Europe/Brussels")),
    end=dt.datetime(2025, 12, 31, tzinfo=ZoneInfo("Europe/Brussels")),
)

timestamp    provider                                  \
                             consumption             fixed         total   
                                  energy   total fixed_fee total   total   
0  2025-01-01 00:00:00+01:00      803.52  803.52      10.0  10.0  813.52   
1  2025-02-01 00:00:00+01:00      725.76  725.76      10.0  10.0  735.76   
2  2025-03-01 00:00:00+01:00      803.52  803.52      10.0  10.0  813.52   
3  2025-04-01 00:00:00+01:00      777.60  777.60      10.0  10.0  787.60   
4  2025-05-01 00:00:00+01:00      803.52  803.52      10.0  10.0  813.52   
5  2025-06-01 00:00:00+01:00      777.60  777.60      10.0  10.0  787.60   
6  2025-07-01 00:00:00+01:00      803.52  803.52      10.0  10.0  813.52   
7  2025-08-01 00:00:00+01:00      803.52  803.52      10.0  10.0  813.52   
8  2025-09-01 00:00:00+01:00      777.60  777.60      10.0  10.0  787.60   
9  2025-10-01 00:00:00+01:00      803.52  803.52      10.0  10.0  813.52   
10 2025-11-01 00:00:00+01:00      777.60  777.60      10.0  10.0  787.60   
11 2025-12-01 00:00:00+01:00      803.52  803.52      10.0  10.0  813.52   

   distributor                                                                \
   consumption               capacity            fixed                 total   
        energy       total      total data_manangement     total       total   
0   525.886877  525.886877  11.092133         1.487151  1.487151  538.466160   
1   474.994598  474.994598  11.092133         1.343233  1.343233  487.429964   
2   525.886877  525.886877  11.092133         1.487151  1.487151  538.466160   
3   508.922784  508.922784  11.092133         1.439178  1.439178  521.454095   
4   525.886877  525.886877  11.092133         1.487151  1.487151  538.466160   
5   508.922784  508.922784  11.104060         1.439178  1.439178  521.466022   
6   525.886877  525.886877  11.138350         1.487151  1.487151  538.512378   
7   525.886877  525.886877  11.192971         1.487151  1.487151  538.566998   
8   508.922784  508.922784  11.266127         1.439178  1.439178  521.628089   
9   525.886877  525.886877  11.356231         1.487151  1.487151  538.730259   
10  508.922784  508.922784  11.461871         1.439178  1.439178  521.823833   
11  525.886877  525.886877  11.461871         1.487151  1.487151  538.835898   

          fees                                                   taxes  \
   consumption                                       total       total   
        excise energy_contribution       total       total       total   
0   406.114744           17.196221  423.310965  423.310965  107.872585   
1   366.813317           15.532070  382.345388  382.345388   97.678243   
2   406.114744           17.196221  423.310965  423.310965  107.872585   
3   393.014268           16.641504  409.655772  409.655772  104.474471   
4   406.114744           17.196221  423.310965  423.310965  107.872585   
5   393.014268           16.641504  409.655772  409.655772  104.475902   
6   406.114744           17.196221  423.310965  423.310965  107.878131   
7   406.114744           17.196221  423.310965  423.310965  107.884685   
8   393.014268           16.641504  409.655772  409.655772  104.495350   
9   406.114744           17.196221  423.310965  423.310965  107.904276   
10  393.014268           16.641504  409.655772  409.655772  104.518839   
11  406.114744           17.196221  423.310965  423.310965  107.916953   

          total  
          total  
          total  
0   1905.748994  
1   1725.648961  
2   1905.748994  
3   1845.715649  
4   1905.748994  
5   1845.740935  
6   1905.846974  
7   1905.962769  
8   1846.084517  
9   1906.308882  
10  1846.499493  
11  1906.532838

As you can see in the above example, while the capacity was constant in 2025, the first months are still cheaper then the last, as the cost of the capacity tariff is based on the consumption of the previous 12 months, which includes the cheaper months of 2024.

### Injection

Aside from a consumption timeseries, you can also provide injection as a separate timeseries, as these are often measured independently and have different tariffs. If you provide injection data, the cost of the injection will be calculated separately and subtracted from the cost of the consumption, as injection is essentially negative consumption.

In [4]:
from energy_cost import PowerDirection

injection = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    ),
    direction=PowerDirection.INJECTION,
)

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption, injection], resolution=isodate.Duration(years=1))

timestamp    provider                                       \
                            consumption         injection              fixed   
                                 energy   total    energy    total fixed_fee   
0 2024-01-01 00:00:00+01:00      702.72  702.72  -140.544 -140.544       0.0   
1 2025-01-01 00:00:00+01:00      630.72  630.72  -105.120 -105.120     120.0   
2 2026-01-01 00:00:00+01:00      135.96  135.96   -28.325  -28.325     180.0   

                  distributor                                                  \
            total consumption                capacity            fixed          
   total    total      energy       total       total data_manangement  total   
0    0.0  562.176  279.535692  279.535692   98.517173            14.28  14.28   
1  120.0  645.600  412.792925  412.792925  133.105596            17.51  17.51   
2  180.0  287.635   59.240491   59.240491   33.875614            17.85  17.85   

                     fees                                              \
        total consumption                                       total   
        total      excise energy_contribution       total       total   
0  392.332864  333.651456           13.535090  347.186546  347.186546   
1  563.408521  332.739840           13.498109  346.237949  346.237949   
2  110.966105   53.794840            2.182271   55.977111   55.977111   

        taxes        total  
        total        total  
        total        total  
0   84.869555  1499.362138  
1  109.551724  1935.413790  
2   41.178230   727.482059

### Expanding the MeterType level

By default the output collapses the `MeterType` level, summing across meter types (e.g. `single_rate`, `tou_peak`) for a cleaner 3-level output: `(TariffCategory, CostGroup, cost_type)`. If you need to distinguish between meter types, pass `expand_meter_types=True` to get a 4-level output: `(TariffCategory, CostGroup, MeterType, cost_type)`.

In [5]:
result = contract.calculate_cost(
    [consumption, injection], resolution=isodate.Duration(years=1), expand_meter_types=True
)
result

timestamp    provider                               \
                            consumption           injection            
                            single_rate         single_rate            
                                 energy   total      energy    total   
0 2024-01-01 00:00:00+01:00      702.72  702.72    -140.544 -140.544   
1 2025-01-01 00:00:00+01:00      630.72  630.72    -105.120 -105.120   
2 2026-01-01 00:00:00+01:00      135.96  135.96     -28.325  -28.325   

                            distributor                          \
      fixed           total consumption                capacity   
        all             all single_rate                     all   
  fixed_fee  total    total      energy       total       total   
0       0.0    0.0  562.176  279.535692  279.535692   98.517173   
1     120.0  120.0  645.600  412.792925  412.792925  133.105596   
2     180.0  180.0  287.635   59.240491   59.240491   33.875614   

                                             fees                      \
             fixed              total consumption                       
               all                all single_rate                       
  data_manangement  total       total      excise energy_contribution   
0            14.28  14.28  392.332864  333.651456           13.535090   
1            17.51  17.51  563.408521  332.739840           13.498109   
2            17.85  17.85  110.966105   53.794840            2.182271   

                                taxes        total  
                    total       total        total  
                      all         all          all  
        total       total       total        total  
0  347.186546  347.186546   84.869555  1499.362138  
1  346.237949  346.237949  109.551724  1935.413790  
2   55.977111   55.977111   41.178230   727.482059